In [0]:
%python
bootstrap_servers = "pkc-zgp5j7.us-south1.gcp.confluent.cloud:9092"
api_key = "ROGKND4WCOI7K25A"
api_secret = "cfltLg1Ur9f9ITypS21Saxwe8gOK6bkVynW4D+R++NrMuZalqfU4Vgs62IpqyMDA"
topic = "credit_card_transactions"

jaas_config=f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="{api_key}" password="{api_secret}";'



df_stream = spark.readStream.format("kafka")\
                         .option("kafka.bootstrap.servers", bootstrap_servers)\
                         .option("subscribe",topic)\
                         .option("kafka.security.protocol", "SASL_SSL")\
                         .option("kafka.sasl.jaas.config", jaas_config)\
                         .option("kafka.sasl.mechanism", "PLAIN")\
                         .option("startingOffsets", "earliest")\
                         .load()
                       
                    

In [0]:
%python
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
%python
df_stream = df_stream.withColumn("key", col("key").cast(StringType())).withColumn("value", col("value").cast(StringType()))

In [0]:
%python
df_stream.writeStream.format("delta")\
         .outputMode("append")\
         .option("checkpointLocation", "/Volumes/fraud_kafka_project/bronze/fraud_kafka_checkpoint")\
         .trigger(availableNow=True)\
         .toTable("fraud_kafka_project.bronze.test")
       

In [0]:
%python
df = spark.sql("select * from fraud_kafka_project.bronze.test")

In [0]:
%python
schema = StructType([
        StructField("transaction_id", StringType(), False),
        StructField("customer_id", StringType(), False),
        StructField("card_number", StringType(), False),
        StructField("merchant_id", StringType(), False),
        StructField("merchant_name", StringType(), True),
        StructField("merchant_category", StringType(), True),
        StructField("amount", DoubleType(), False),
        StructField("currency", StringType(), True),
        StructField("transaction_type", StringType(), True),
        StructField("payment_channel", StringType(), True),
        StructField("device_id", StringType(), True),
        StructField("city", StringType(), True),
        StructField("country", StringType(), True),
        StructField("transaction_timestamp", StringType(), True),
        StructField("is_international", BooleanType(), True),
        StructField("status", StringType(), True),
    ])

In [0]:
%python
df = df.withColumn("value", from_json(col("value"), schema))


In [0]:
%python
df = df.select(
    col("key").alias("transaction_id"),
    col("value.*"),
    col("topic").alias("topic"),
    col("partition").alias("partition"),
    col("offset").alias("offset"),
    col("timestampType").alias("timestampType"),
    col("timestamp").alias("transaction_timestamp")
)

In [0]:
%python
display(df)

In [0]:
select * from fraud_kafka_project.silver.silver_credit_card_transactions

In [0]:
DELETE FROM fraud_kafka_project.silver.cleaned_fraud_watchlist WHERE 1=1

In [0]:
drop table fraud_kafka_project.silver.cleaned_fraud_watchlist